# CAMEL Ambient Solar Wind Validation

## Overview

This notebook demonstrates how to validate ambient solar wind models against in situ measurements near Earth. The workflow is based on the **CAMEL (Comprehensive Assessment of Models and Events using Library tools)** framework developed at the [Community Coordinated Modeling Center (CCMC)](https://ccmc.gsfc.nasa.gov/).

This effort is part of the international [ISWAT H1-01 Team](https://iswat-cospar.org/h1-01), which aims to establish a unified validation approach for solar wind models across the heliophysics and space weather communities.

The validation procedures presented in this notebook series are also available through an interactive web application built with Plotly Dash and Python. The **Ambient Solar Wind Validation Dashboard** can be accessed here:

- [CCMC CAMEL Ambient Solar Wind Validation Dashboard](https://ccmc.gsfc.nasa.gov/camel/AmbientSolarWind/home)

These notebooks provide a reproducible version of the same validation workflow and are intended to help users explore, reproduce, and extend the validation procedures of the CCMC CAMEL dashboard.

**Notebook version:** 1.0

---

## Notebook Scope

### 02 Point-to-Point Metrics

This notebook:
- loads the preprocessed example datasets generated in `01_api_and_preprocessing.ipynb`
- computes descriptive statistics and standard point-to-point validation metrics
- reproduces the point-to-point visualization examples used in the dashboard workflow

---

## Expected Inputs

This notebook expects the following processed datasets:

```text
../example_data/processed/
├── omni_obs.csv
├── ambsowi_mod.csv
├── bravda_mod.csv
└── p27_mod.csv
```

## Imports

First, we import the metric and visual diagnostic functions from `utils/`. Note that these functions are an exact copy of the functions that are used in the CAMEL Ambient Solar Wind Dashboard. 


In [ ]:
import sys
from pathlib import Path

project_root = ".."
utils_path = "../utils"

if project_root not in sys.path:
    sys.path.append(project_root)
if utils_path not in sys.path:
    sys.path.append(utils_path)

import numpy as np
import pandas as pd
from IPython.display import display

from skill_scores import *
from utils.visualizations import *

## Load the processed example datasets

These CSV files are assumed to have already been preprocessed and aligned in `01_api_and_preprocessing.ipynb`. Note that if you want to change the selected time window for validation you need to go back to `01_api_and_preprocessing.ipynb`.


In [ ]:
data_dir = Path("../example_data/processed")

omni_obs = pd.read_csv(data_dir / "omni_obs.csv", parse_dates=["Time"], index_col="Time")
ambsowi_mod = pd.read_csv(data_dir / "ambsowi_mod.csv", parse_dates=["Time"], index_col="Time")
bravda_mod = pd.read_csv(data_dir / "bravda_mod.csv", parse_dates=["Time"], index_col="Time")
p27_mod = pd.read_csv(data_dir / "p27_mod.csv", parse_dates=["Time"], index_col="Time")

print("Observed shape:   ", omni_obs.shape)
print("AmbSoWi-ML shape: ", ambsowi_mod.shape)
print("BRaVDA shape:     ", bravda_mod.shape)
print("Persistence shape:", p27_mod.shape)


## Compute standard point-to-point metrics

We compute the same core metrics used in the dashboard workflow:
- Mean Error (ME)
- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- Correlation Coefficient (CC)


In [ ]:
def compute_metric_table(models: dict, observed: pd.DataFrame) -> pd.DataFrame:
    rows = {}
    for name, model_df in models.items():
        rows[name] = {
            "ME": float(meScore(model_df, observed)),
            "MAE": float(maeScore(model_df, observed)),
            "MSE": float(meanSquaredError(model_df, observed)),
            "RMSE": float(rmsScore(model_df, observed)),
            "CC": float(ccScore(model_df, observed)),
        }
    return pd.DataFrame.from_dict(rows, orient="index")

models = {
    "AmbSoWi-ML": ambsowi_mod,
    "BRaVDA": bravda_mod,
    "Persistence (27d)": p27_mod,
}

metrics_table = compute_metric_table(models, omni_obs)
display(metrics_table.round(2))


## Compute descriptive statistics

We use the same `descriptiveStats` helper used in the dashboard to summarize the distributions of observations and model outputs.


In [ ]:
stats_df = pd.DataFrame({
    "Observed": descriptiveStats(omni_obs["Flow_Speed"]),
    "AmbSoWi-ML": descriptiveStats(ambsowi_mod["Flow_Speed"]),
    "BRaVDA": descriptiveStats(bravda_mod["Flow_Speed"]),
    "Persistence (27d)": descriptiveStats(p27_mod["Flow_Speed"]),
})

display(stats_df)


## Prepare inputs for the visual diagnostics

Convert the processed time series into the long-format structure expected by the CAMEL dashboard visualization functions.

In [ ]:
def to_pred(df: pd.DataFrame, name: str) -> pd.DataFrame:
    d = df.copy().reset_index()
    d = d.rename(columns={"Flow_Speed": "Model"})
    d["Name"] = name
    return d[["Time", "Model", "Name"]]

def to_obs(df: pd.DataFrame, name: str) -> pd.DataFrame:
    d = df.copy().reset_index()
    d = d.rename(columns={"Flow_Speed": "Obs"})
    d["Name"] = name
    return d[["Time", "Obs", "Name"]]

observed_data = to_obs(omni_obs, "OMNI2 | Hourly averaged")

predicted_ambsowi = to_pred(ambsowi_mod, "AmbSoWi-ML")
predicted_bravda = to_pred(bravda_mod, "BRaVDA")
baseline_data = to_pred(p27_mod, "Persistence (27-days)")

## Visual Diagnostics for Point-to-Point Metrics

These cells mirror the examples from the original notebook:
- time series plot
- raincloud plot
- scatter plot
- residual plot
- RMSE timeline

The visualizations below use **AmbSoWi-ML** against **OMNI** with the **27-day persistence** baseline.


In [ ]:
fig = create_line_chart(predicted_ambsowi, observed_data, baseline_data)
fig.show()

In [ ]:
fig = create_raincloud_plot(predicted_ambsowi, observed_data)
fig.update_layout(width=500, height=500)
fig.show()

In [ ]:
fig = create_scatter_plot(predicted_ambsowi, observed_data)
fig.update_layout(width=500, height=500)
fig.show()

In [ ]:
fig = create_residual_plot(predicted_ambsowi, observed_data)
fig.update_layout(width=500, height=500)
fig.show()

In [ ]:
fig = create_rmse_timeline(predicted_ambsowi, observed_data, baseline_data, normalize_time="naive_utc", floor="H")
fig.update_layout(width=1300, height=500)
fig.show()

## PDF export example

If `kaleido` is installed, all figures can be exported as vector PDFs.

In [ ]:
import os

out_dir = "../results/output_pdfs"
os.makedirs(out_dir, exist_ok=True)

fig = create_scatter_plot(predicted_ambsowi, observed_data)
fig.update_layout(width=500, height=500)

pdf_path = os.path.join(out_dir, "scatter_plot_ambsowi.pdf")

# Requires kaleido:
fig.write_image(pdf_path)

print("Saved:", pdf_path)

Save all visualizations generated in this notebook as PDF files in `../results/output_pdfs/`. This requires `kaleido`.

In [ ]:
from pathlib import Path

out_dir = Path("../results/output_pdfs")
out_dir.mkdir(parents=True, exist_ok=True)

figures = {
    "line_chart_ambsowi": create_line_chart(
        predicted_ambsowi,
        observed_data,
        baseline_data,
    ),
    "raincloud_plot_ambsowi": create_raincloud_plot(
        predicted_ambsowi,
        observed_data,
    ),
    "scatter_plot_ambsowi": create_scatter_plot(
        predicted_ambsowi,
        observed_data,
    ),
    "residual_plot_ambsowi": create_residual_plot(
        predicted_ambsowi,
        observed_data,
    ),
    "rmse_timeline_ambsowi": create_rmse_timeline(
        predicted_ambsowi,
        observed_data,
        baseline_data,
        normalize_time="naive_utc",
        floor="H",
    )
}

figure_sizes = {
    "line_chart_ambsowi": (1200, 500),
    "raincloud_plot_ambsowi": (500, 500),
    "scatter_plot_ambsowi": (500, 500),
    "residual_plot_ambsowi": (500, 500),
    "rmse_timeline_ambsowi": (1200, 500),
}

for name, fig in figures.items():
    width, height = figure_sizes[name]
    fig.update_layout(width=width, height=height)

    pdf_path = out_dir / f"{name}.pdf"

    # Requires kaleido:
    fig.write_image(pdf_path)

    print("Saved:", pdf_path)